# Official YouTube Data API v3 explorer

This notebook uses documented, read-only YouTube Data API v3 endpoints. Create a Google Cloud project, enable **YouTube Data API v3**, and create an API key before running it.

In [ ]:
%pip -q install pandas requests

from getpass import getpass
from urllib.parse import parse_qs, urlparse

import pandas as pd
import requests

API_KEY = getpass('YouTube Data API key: ')
BASE_URL = 'https://www.googleapis.com/youtube/v3'

def youtube_get(resource, **params):
    response = requests.get(
        f'{BASE_URL}/{resource}',
        params={'key': API_KEY, **params},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()

def video_id_from_url(url):
    parsed = urlparse(url)
    host = (parsed.hostname or '').lower()
    parts = [part for part in parsed.path.split('/') if part]
    if host == 'youtu.be':
        return parts[0] if parts else None
    if host.endswith('youtube.com'):
        if parsed.path == '/watch':
            return parse_qs(parsed.query).get('v', [None])[0]
        if len(parts) >= 2 and parts[0] in {'shorts', 'embed', 'live'}:
            return parts[1]
    return None


## 1. `videos.list`: metadata, duration, statistics, and live details

In [ ]:
VIDEO_URL = 'https://www.youtube.com/watch?v=MM-Qhlxf1pM'
VIDEO_ID = video_id_from_url(VIDEO_URL)

video_response = youtube_get(
    'videos',
    part='snippet,contentDetails,statistics,status,liveStreamingDetails',
    id=VIDEO_ID,
)
video = video_response['items'][0]
pd.json_normalize(video)

## 2. `channels.list`: the video's channel and its uploads playlist

The channel response contains the playlist ID that represents all public uploads.

In [ ]:
CHANNEL_ID = video['snippet']['channelId']
channel_response = youtube_get(
    'channels',
    part='snippet,statistics,contentDetails,status',
    id=CHANNEL_ID,
)
channel = channel_response['items'][0]
UPLOADS_PLAYLIST_ID = channel['contentDetails']['relatedPlaylists']['uploads']
print('Channel:', channel['snippet']['title'])
print('Uploads playlist:', UPLOADS_PLAYLIST_ID)
pd.json_normalize(channel)

## 3. `playlistItems.list`: recent public uploads

Change `maxResults` or use `nextPageToken` from the response to paginate.

In [ ]:
uploads_response = youtube_get(
    'playlistItems',
    part='snippet,contentDetails,status',
    playlistId=UPLOADS_PLAYLIST_ID,
    maxResults=25,
)
uploads = pd.json_normalize(uploads_response['items'])
uploads[['contentDetails.videoId', 'snippet.publishedAt', 'snippet.title', 'snippet.videoOwnerChannelTitle']].head(25)

## 4. `search.list`: query discovery

Search is useful for exploration but costs more quota than the direct ID-based endpoints. Use direct video/channel/playlist IDs once you know them.

In [ ]:
QUERY = 'humanitarian response'
search_response = youtube_get(
    'search',
    part='snippet',
    q=QUERY,
    type='video',
    order='date',
    maxResults=25,
)
search_results = pd.json_normalize(search_response['items'])
search_results[['id.videoId', 'snippet.publishedAt', 'snippet.channelTitle', 'snippet.title', 'snippet.description']].head(25)

## 5. `commentThreads.list`: top-level public comments

Some videos have comments disabled. The response includes top-level comments; replies may require a separate `comments.list` request.

In [ ]:
try:
    comments_response = youtube_get(
        'commentThreads',
        part='snippet,replies',
        videoId=VIDEO_ID,
        maxResults=25,
        order='relevance',
        textFormat='plainText',
    )
    comments = pd.json_normalize(comments_response['items'])
    display(comments[[
        'id',
        'snippet.topLevelComment.snippet.authorDisplayName',
        'snippet.topLevelComment.snippet.publishedAt',
        'snippet.topLevelComment.snippet.textDisplay',
        'snippet.totalReplyCount',
    ]].head(25))
except requests.HTTPError as error:
    print(f'Comments unavailable for this video: {error}')

## Captions boundary

The official API's `captions.list` can identify caption tracks, but caption download requires OAuth authorization from the video owner. It cannot be used to download captions from arbitrary public videos with an API key.